In [ ]:
#

import os
import re
import json
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from matplotlib.ticker import MaxNLocator
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

# ================= CONFIG =================
LABEL_CSV = "../LiverFibrosisCohort/LiverFibrosisCohort_MasterTable_Proccessed_527.csv"

RAD_ROOT = "../LiverFibrosisCohort/Data/Radiomics"
DEEP_ROOT = "../LiverFibrosisCohort/Data/deep_medsiglip_v2"

MODALITIES = ["ADC", "T1", "T2"]
DEV_SITES = ["CCHMC", "WISC", "NYU"]
EXTERNAL_SITE = "MICH"

ID_COLUMN = "Image ID"
LABEL_COLUMN = "Fibrosis_Label"

BRANCH_CONFIG = {
    "ADC": {"rad_hidden": 16, "deep_hidden": 64},
    "T1": {"rad_hidden": 8, "deep_hidden": 128},
    "T2": {"rad_hidden": 8, "deep_hidden": 128},
}

FUSION_HEAD_OPTIONS = [
    [32],
    [16],
    [8],
    [],
]

EPOCHS = 100
LR = 4e-5
N_SPLITS = 5
SEED = 42
THRESHOLD = 0.47  # prespecified fixed threshold
BATCH_SIZE = 32
BOOTSTRAP_N = 500
BOOTSTRAP_CI = 95

CV_EARLY_STOPPING_VAL_FRAC = 0.20
FINAL_RETRAIN_VAL_FRAC = 0.20
EARLY_STOPPING_PATIENCE = 15
MIN_EPOCHS = 10

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR = "../Results/multimodal_fusion_rad_deep_results"
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
LEARNING_CURVE_CSV_DIR = os.path.join(OUTPUT_DIR, "learning_curve_csv")
FINAL_RETRAIN_DIR = os.path.join(PLOT_DIR, "final_retrain")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(LEARNING_CURVE_CSV_DIR, exist_ok=True)
os.makedirs(FINAL_RETRAIN_DIR, exist_ok=True)

# ================= PUBLICATION PLOT CONFIG =================
PUB_FIG_W = 4.8
PUB_FIG_H = 3.8
PUB_TITLE_FS = 13
PUB_LABEL_FS = 12
PUB_TICK_FS = 10
PUB_LEGEND_FS = 10
PUB_LINE_W = 2.2
PUB_SPINE_W = 1.0
PUB_MARKER_SIZE = 45


# ================= REPRODUCIBILITY =================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)


# ================= ID HELPERS =================
def normalize_id(pid):
    pid = str(pid)

    m = re.match(r"PT-(\d{3}-\d{4})-", pid)
    if m:
        return m.group(1)

    m = re.match(r"PT-(M\d+)-", pid)
    if m:
        return m.group(1)

    return pid


def get_site(pid):
    pid = str(pid).strip().upper()

    if re.match(r"^\d{3}-\d{4}$", pid):
        return "CCHMC"
    elif pid.startswith("GJ"):
        return "WISC"
    elif pid.startswith("C"):
        return "MICH"
    elif re.match(r"^\d+$", pid):
        return "NYU"
    else:
        return "UNKNOWN"


# ================= LABEL =================
def binarize_fibrosis(x):
    if pd.isna(x):
        return np.nan

    x = str(x)
    m = re.search(r"\d+", x)
    if not m:
        return np.nan

    val = int(m.group())
    if val in [0, 1]:
        return 0
    elif val in [2, 3, 4]:
        return 1

    return np.nan


# ================= DATA LOADING =================
def load_labels():
    labels_df = pd.read_csv(LABEL_CSV)
    labels_df = labels_df.rename(columns={ID_COLUMN: "id"})

    labels_df["id"] = labels_df["id"].astype(str).apply(normalize_id)
    labels_df["label"] = labels_df[LABEL_COLUMN].apply(binarize_fibrosis)

    labels_df = labels_df.dropna(subset=["label"]).copy()
    labels_df["label"] = labels_df["label"].astype(int)
    labels_df["site"] = labels_df["id"].apply(get_site)

    labels_df = labels_df[["id", "label", "site"]]
    labels_df = labels_df.drop_duplicates(subset=["id"]).reset_index(drop=True)

    return labels_df


def load_radiomics(modality):
    liver_path = os.path.join(RAD_ROOT, f"{modality}_liver_features.csv")
    spleen_path = os.path.join(RAD_ROOT, f"{modality}_spleen_features.csv")

    if not os.path.exists(liver_path):
        raise FileNotFoundError(f"Missing file: {liver_path}")
    if not os.path.exists(spleen_path):
        raise FileNotFoundError(f"Missing file: {spleen_path}")

    liver_df = pd.read_csv(liver_path)
    spleen_df = pd.read_csv(spleen_path)

    liver_df["ID"] = liver_df["ID"].astype(str).apply(normalize_id)
    spleen_df["ID"] = spleen_df["ID"].astype(str).apply(normalize_id)

    liver_df = liver_df.rename(columns={"ID": "id"})
    spleen_df = spleen_df.rename(columns={"ID": "id"})

    rad_df = liver_df.merge(spleen_df, on="id", suffixes=("_liver", "_spleen"))
    rad_df = rad_df.drop_duplicates(subset=["id"]).reset_index(drop=True)

    new_cols = {}
    for c in rad_df.columns:
        if c != "id":
            new_cols[c] = f"{modality}_RAD_{c}"
    rad_df = rad_df.rename(columns=new_cols)

    return rad_df


def load_deep(modality):
    feat_path = os.path.join(DEEP_ROOT, f"{modality}_medsiglip.csv")
    if not os.path.exists(feat_path):
        raise FileNotFoundError(f"Missing file: {feat_path}")

    deep_df = pd.read_csv(feat_path)
    deep_df["id"] = deep_df["id"].astype(str).apply(normalize_id)
    deep_df = deep_df.drop_duplicates(subset=["id"]).reset_index(drop=True)

    new_cols = {}
    for c in deep_df.columns:
        if c != "id":
            new_cols[c] = f"{modality}_DEEP_{c}"
    deep_df = deep_df.rename(columns=new_cols)

    return deep_df


def build_multimodal_table(labels_df):
    merged = labels_df.copy()
    branch_cols = {}

    for modality in MODALITIES:
        rad_df = load_radiomics(modality)
        deep_df = load_deep(modality)

        merged = merged.merge(rad_df, on="id", how="left")
        merged = merged.merge(deep_df, on="id", how="left")

        rad_cols = [c for c in merged.columns if c.startswith(f"{modality}_RAD_")]
        deep_cols = [c for c in merged.columns if c.startswith(f"{modality}_DEEP_")]

        if len(rad_cols) == 0:
            raise ValueError(f"No radiomics columns found for {modality}")
        if len(deep_cols) == 0:
            raise ValueError(f"No deep columns found for {modality}")

        branch_cols[modality] = {
            "rad": rad_cols,
            "deep": deep_cols,
        }

    return merged, branch_cols


# ================= PREPROCESS HELPERS =================
def fit_mean_imputer(X_train):
    means = np.nanmean(X_train, axis=0)
    means = np.where(np.isnan(means), 0.0, means)
    return means


def apply_mean_imputer(X, means):
    X_out = X.copy()
    inds = np.where(np.isnan(X_out))
    X_out[inds] = means[inds[1]]
    return X_out


def fit_branch_preprocessor(X_train):
    X_train = X_train.astype(np.float32)
    X_train[np.isinf(X_train)] = np.nan

    means = fit_mean_imputer(X_train)
    X_train = apply_mean_imputer(X_train, means)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)

    return X_train, {"means": means, "scaler": scaler}


def transform_branch(X, preprocessor):
    X = X.astype(np.float32)
    X[np.isinf(X)] = np.nan
    X = apply_mean_imputer(X, preprocessor["means"])
    X = preprocessor["scaler"].transform(X)
    return X


def get_branch_input_dims(branch_cols):
    branch_input_dims = {}
    for modality in MODALITIES:
        branch_input_dims[modality] = {
            "rad_dim": len(branch_cols[modality]["rad"]),
            "deep_dim": len(branch_cols[modality]["deep"]),
        }
    return branch_input_dims


def fit_preprocessors_and_make_tensors(train_df, branch_cols):
    X_train_dict = {}
    preprocessors = {}

    for modality in MODALITIES:
        rad_cols = branch_cols[modality]["rad"]
        deep_cols = branch_cols[modality]["deep"]

        Xr_tr = train_df[rad_cols].to_numpy(dtype=np.float32)
        Xd_tr = train_df[deep_cols].to_numpy(dtype=np.float32)

        Xr_tr, rad_pre = fit_branch_preprocessor(Xr_tr)
        Xd_tr, deep_pre = fit_branch_preprocessor(Xd_tr)

        preprocessors[modality] = {
            "rad": rad_pre,
            "deep": deep_pre,
        }

        X_train_dict[modality] = {
            "rad": torch.tensor(Xr_tr, dtype=torch.float32, device=DEVICE),
            "deep": torch.tensor(Xd_tr, dtype=torch.float32, device=DEVICE),
        }

    return X_train_dict, preprocessors


def transform_df_to_tensors(df, branch_cols, preprocessors):
    X_dict = {}

    for modality in MODALITIES:
        rad_cols = branch_cols[modality]["rad"]
        deep_cols = branch_cols[modality]["deep"]

        Xr = df[rad_cols].to_numpy(dtype=np.float32)
        Xd = df[deep_cols].to_numpy(dtype=np.float32)

        Xr = transform_branch(Xr, preprocessors[modality]["rad"])
        Xd = transform_branch(Xd, preprocessors[modality]["deep"])

        X_dict[modality] = {
            "rad": torch.tensor(Xr, dtype=torch.float32, device=DEVICE),
            "deep": torch.tensor(Xd, dtype=torch.float32, device=DEVICE),
        }

    return X_dict


# ================= METRICS =================
def safe_roc_auc(y_true, y_prob):
    try:
        return float(roc_auc_score(y_true, y_prob))
    except ValueError:
        return np.nan


def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    if np.isnan(sensitivity) or np.isnan(specificity):
        balanced_acc = np.nan
    else:
        balanced_acc = float((sensitivity + specificity) / 2.0)

    auc = safe_roc_auc(y_true, y_prob)

    return {
        "balanced_accuracy": balanced_acc,
        "auc": auc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def summarize_bootstrap_distribution(values, ci=95):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "n_valid": 0,
        }

    alpha = (100 - ci) / 2.0

    if len(values) == 1:
        return {
            "mean": float(values[0]),
            "std": 0.0,
            "ci_lower": float(values[0]),
            "ci_upper": float(values[0]),
            "n_valid": 1,
        }

    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values, ddof=1)),
        "ci_lower": float(np.percentile(values, alpha)),
        "ci_upper": float(np.percentile(values, 100 - alpha)),
        "n_valid": int(len(values)),
    }


def bootstrap_binary_metrics(y_true, y_prob, threshold=0.5, n_bootstrap=500, ci=95, seed=42):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    point_metrics = compute_binary_metrics(y_true, y_prob, threshold=threshold)

    rng = np.random.default_rng(seed)
    n = len(y_true)

    metric_names = ["balanced_accuracy", "auc", "sensitivity", "specificity"]
    boot_store = {k: [] for k in metric_names}

    for _ in range(n_bootstrap):
        sample_idx = rng.integers(0, n, size=n)
        boot_metrics = compute_binary_metrics(
            y_true[sample_idx],
            y_prob[sample_idx],
            threshold=threshold,
        )
        for metric_name in metric_names:
            boot_store[metric_name].append(boot_metrics[metric_name])

    out = {
        "n_bootstrap": int(n_bootstrap),
        "bootstrap_ci": int(ci),
    }

    for metric_name, metric_value in point_metrics.items():
        out[f"{metric_name}_point"] = metric_value

    for metric_name in metric_names:
        summary = summarize_bootstrap_distribution(boot_store[metric_name], ci=ci)
        out[f"{metric_name}"] = summary["mean"]
        out[f"{metric_name}_bootstrap_mean"] = summary["mean"]
        out[f"{metric_name}_bootstrap_std"] = summary["std"]
        out[f"{metric_name}_bootstrap_ci_lower"] = summary["ci_lower"]
        out[f"{metric_name}_bootstrap_ci_upper"] = summary["ci_upper"]
        out[f"{metric_name}_bootstrap_n_valid"] = summary["n_valid"]

    for count_name in ["tp", "tn", "fp", "fn"]:
        out[count_name] = point_metrics[count_name]
        out[f"{count_name}_point"] = point_metrics[count_name]

    return out


def mean_std(series):
    vals = pd.Series(series).dropna().values
    if len(vals) == 0:
        return np.nan, np.nan
    if len(vals) == 1:
        return float(vals[0]), 0.0
    return float(np.mean(vals)), float(np.std(vals, ddof=1))


# ================= MODEL =================
class BranchEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class MultiModalFusionMLP(nn.Module):
    def __init__(self, branch_input_dims, branch_hidden_config, fusion_hidden_dims=None):
        super().__init__()

        if fusion_hidden_dims is None:
            fusion_hidden_dims = []

        self.modalities = list(branch_input_dims.keys())
        self.rad_encoders = nn.ModuleDict()
        self.deep_encoders = nn.ModuleDict()

        total_fused_dim = 0
        for modality in self.modalities:
            rad_dim = branch_input_dims[modality]["rad_dim"]
            deep_dim = branch_input_dims[modality]["deep_dim"]

            rad_hidden = branch_hidden_config[modality]["rad_hidden"]
            deep_hidden = branch_hidden_config[modality]["deep_hidden"]

            self.rad_encoders[modality] = BranchEncoder(rad_dim, rad_hidden)
            self.deep_encoders[modality] = BranchEncoder(deep_dim, deep_hidden)

            total_fused_dim += rad_hidden + deep_hidden

        layers = []
        in_dim = total_fused_dim

        if len(fusion_hidden_dims) == 0:
            layers.append(nn.Linear(in_dim, 1))
        else:
            for h in fusion_hidden_dims:
                layers.append(nn.Linear(in_dim, h))
                layers.append(nn.ReLU())
                in_dim = h
            layers.append(nn.Linear(in_dim, 1))

        self.classifier = nn.Sequential(*layers)

    def forward(self, x_dict):
        feats = []
        for modality in self.modalities:
            x_rad = x_dict[modality]["rad"]
            x_deep = x_dict[modality]["deep"]

            h_rad = self.rad_encoders[modality](x_rad)
            h_deep = self.deep_encoders[modality](x_deep)

            feats.append(h_rad)
            feats.append(h_deep)

        h = torch.cat(feats, dim=1)
        out = self.classifier(h)
        return out.squeeze(1)


# ================= PLOTTING =================
def fusion_head_to_label(fusion_hidden_dims):
    if len(fusion_hidden_dims) == 0:
        return "direct"
    return "→".join(str(x) for x in fusion_hidden_dims)


def sanitize_for_filename(text):
    text = str(text)
    text = text.replace("→", "_to_")
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text


def plot_metric_bar(results_df, mean_col, std_col, ylabel, title, out_path):
    plot_df = results_df.copy().reset_index(drop=True)

    x = np.arange(len(plot_df))
    y = plot_df[mean_col].values.astype(float)
    yerr = plot_df[std_col].fillna(0).values.astype(float)
    labels = ["direct" if v == "direct" else v for v in plot_df["fusion_head_label"].tolist()]

    plt.figure(figsize=(6, 5))
    bars = plt.bar(x, y, yerr=yerr, capsize=5, width=0.5)

    plt.xticks(x, labels, fontsize=18)
    plt.yticks(fontsize=18)
    plt.ylabel(ylabel, fontsize=18)
    plt.xlabel("Fusion Head", fontsize=18)
    plt.title(title, fontsize=22)
    plt.ylim(0, min(1.0, np.nanmax(y + yerr) + 0.08))

    for bar, val in zip(bars, y):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{val:.2f}",
            ha="center",
            va="bottom",
            fontsize=18,
        )

    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()


def plot_learning_curve(
    curve_df,
    out_path_png,
    out_path_pdf=None,
    title=None,
    panel_label=None,
):
    curve_df = curve_df.sort_values("epoch").reset_index(drop=True).copy()

    fig, ax = plt.subplots(figsize=(PUB_FIG_W, PUB_FIG_H))

    ax.plot(
        curve_df["epoch"],
        curve_df["train_loss"],
        linewidth=PUB_LINE_W,
        color="black",
        marker="o",
        markersize=3.2,
        markerfacecolor="black",
        markeredgewidth=0.9,
        label="Training loss",
        zorder=2,
    )

    has_val = "val_loss" in curve_df.columns and curve_df["val_loss"].notna().any()
    best_epoch = None
    best_val_loss = None

    if has_val:
        ax.plot(
            curve_df["epoch"],
            curve_df["val_loss"],
            linewidth=PUB_LINE_W,
            color="#D55E00",
            marker="o",
            markersize=3.2,
            markerfacecolor="#D55E00",
            markeredgewidth=0.9,
            label="Validation loss",
            zorder=3,
        )

        best_idx = curve_df["val_loss"].astype(float).idxmin()
        best_row = curve_df.loc[best_idx]
        best_epoch = int(best_row["epoch"])
        best_val_loss = float(best_row["val_loss"])

        ax.axvline(
            best_epoch,
            color="#D55E00",
            linestyle="--",
            linewidth=1.2,
            alpha=0.9,
            zorder=1,
        )
        ax.scatter(
            [best_epoch],
            [best_val_loss],
            s=PUB_MARKER_SIZE,
            color="#D55E00",
            edgecolor="white",
            linewidth=0.8,
            zorder=4,
        )

    if "stopped_epoch" in curve_df.columns and curve_df["stopped_epoch"].notna().any():
        stopped_epoch = int(curve_df["stopped_epoch"].iloc[0])
        if best_epoch is None or stopped_epoch != best_epoch:
            ax.axvline(
                stopped_epoch,
                color="gray",
                linestyle=":",
                linewidth=1.1,
                alpha=0.8,
                zorder=1,
            )

    ax.set_xlabel("Epoch", fontsize=PUB_LABEL_FS)
    ax.set_ylabel("Binary cross-entropy loss", fontsize=PUB_LABEL_FS)

    if title is not None:
        ax.set_title(title, fontsize=PUB_TITLE_FS, pad=8)

    if panel_label is not None:
        ax.text(
            -0.14, 1.03, panel_label,
            transform=ax.transAxes,
            fontsize=PUB_TITLE_FS,
            fontweight="bold",
            va="bottom",
            ha="left",
        )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(PUB_SPINE_W)
    ax.spines["bottom"].set_linewidth(PUB_SPINE_W)

    ax.tick_params(
        axis="both",
        which="major",
        labelsize=PUB_TICK_FS,
        width=PUB_SPINE_W,
        length=4,
        direction="out",
    )

    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xlim(curve_df["epoch"].min(), curve_df["epoch"].max())
    ax.grid(False)

    ax.legend(
        frameon=False,
        fontsize=PUB_LEGEND_FS,
        loc="best",
        handlelength=2.4,
    )

    if best_epoch is not None and best_val_loss is not None:
        annotation = f"Best epoch = {best_epoch}\nBest val loss = {best_val_loss:.3f}"
        ax.text(
            0.98,
            0.98,
            annotation,
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=9,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=2.0),
        )

    fig.tight_layout()
    fig.savefig(out_path_png, dpi=600, bbox_inches="tight", facecolor="white")

    if out_path_pdf is not None:
        fig.savefig(out_path_pdf, bbox_inches="tight", facecolor="white")

    plt.close(fig)


# ================= TRAIN / EVAL =================
def predict_probs(model, X_dict):
    model.eval()
    with torch.no_grad():
        logits = model(X_dict)
        probs = torch.sigmoid(logits).cpu().numpy()
    return probs


def make_train_earlystop_split(df, val_frac, seed):
    try:
        train_df, val_df = train_test_split(
            df,
            test_size=val_frac,
            random_state=seed,
            stratify=df["label"].values.astype(int),
        )
    except ValueError:
        print("Warning: stratified early-stopping split failed. Falling back to unstratified split.")
        train_df, val_df = train_test_split(
            df,
            test_size=val_frac,
            random_state=seed,
            shuffle=True,
        )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def train_model(
    X_train_dict,
    y_train,
    branch_input_dims,
    branch_config,
    fusion_hidden_dims,
    X_val_dict=None,
    y_val=None,
    early_stopping_patience=None,
    min_epochs=1,
):
    y_train_t = torch.tensor(y_train.astype(np.float32), dtype=torch.float32, device=DEVICE)

    y_val_t = None
    if y_val is not None:
        y_val_t = torch.tensor(y_val.astype(np.float32), dtype=torch.float32, device=DEVICE)

    model = MultiModalFusionMLP(
        branch_input_dims=branch_input_dims,
        branch_hidden_config=branch_config,
        fusion_hidden_dims=fusion_hidden_dims,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss()

    curve_rows = []

    use_early_stopping = (
        X_val_dict is not None
        and y_val is not None
        and early_stopping_patience is not None
    )

    best_state_dict = None
    best_epoch = None
    best_val_loss = np.inf
    best_val_auc_at_best_epoch = np.nan
    epochs_without_improvement = 0

    n_train = len(y_train)

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = torch.randperm(n_train, device=DEVICE)
        batch_losses = []

        for start in range(0, n_train, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]

            batch_x_dict = {
                modality: {
                    "rad": X_train_dict[modality]["rad"][idx],
                    "deep": X_train_dict[modality]["deep"][idx],
                }
                for modality in MODALITIES
            }
            batch_y = y_train_t[idx]

            logits = model(batch_x_dict)
            loss = loss_fn(logits, batch_y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_losses.append(float(loss.item()))

        train_loss_epoch = float(np.mean(batch_losses))
        train_loss_std = float(np.std(batch_losses, ddof=1)) if len(batch_losses) > 1 else 0.0

        row = {
            "epoch": epoch,
            "train_loss": train_loss_epoch,
            "train_loss_std": train_loss_std,
            "val_loss": np.nan,
            "val_auc": np.nan,
            "is_best_epoch": 0,
        }

        if X_val_dict is not None and y_val_t is not None:
            model.eval()
            with torch.no_grad():
                val_logits = model(X_val_dict)
                val_loss = loss_fn(val_logits, y_val_t)
                val_probs = torch.sigmoid(val_logits).cpu().numpy()

            val_loss_value = float(val_loss.item())
            val_auc = safe_roc_auc(y_val, val_probs)

            row["val_loss"] = val_loss_value
            row["val_auc"] = val_auc

            if use_early_stopping:
                if best_epoch is None or val_loss_value < best_val_loss:
                    best_val_loss = val_loss_value
                    best_val_auc_at_best_epoch = val_auc
                    best_epoch = epoch
                    best_state_dict = copy.deepcopy(model.state_dict())
                    epochs_without_improvement = 0
                else:
                    epochs_without_improvement += 1

        curve_rows.append(row)

        if use_early_stopping and epoch >= min_epochs and epochs_without_improvement >= early_stopping_patience:
            print(
                f"Early stopping at epoch {epoch} | "
                f"best epoch = {best_epoch} | best val loss = {best_val_loss:.6f}"
            )
            break

    if use_early_stopping and best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    curve_df = pd.DataFrame(curve_rows)

    if use_early_stopping and "val_loss" in curve_df.columns and curve_df["val_loss"].notna().any():
        best_idx = curve_df["val_loss"].astype(float).idxmin()
        curve_df["is_best_epoch"] = 0
        curve_df.loc[best_idx, "is_best_epoch"] = 1

        best_epoch = int(curve_df.loc[best_idx, "epoch"])
        best_val_loss = float(curve_df.loc[best_idx, "val_loss"])
        best_val_auc_at_best_epoch = float(curve_df.loc[best_idx, "val_auc"])
    else:
        best_epoch = int(curve_df["epoch"].iloc[-1])
        curve_df["is_best_epoch"] = 0
        curve_df.loc[curve_df.index[-1], "is_best_epoch"] = 1
        best_val_loss = np.nan
        best_val_auc_at_best_epoch = np.nan

    train_info = {
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss) if np.isfinite(best_val_loss) else np.nan,
        "best_val_auc_at_best_epoch": float(best_val_auc_at_best_epoch) if not np.isnan(best_val_auc_at_best_epoch) else np.nan,
        "stopped_epoch": int(curve_df["epoch"].iloc[-1]),
        "used_early_stopping": bool(use_early_stopping),
        "early_stopping_metric": "val_loss" if use_early_stopping else None,
    }

    return model, curve_df, train_info


def train_eval_multimodal_cv(dev_df, branch_cols, branch_config, fusion_hidden_dims, head_label):
    y = dev_df["label"].values.astype(int)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    fold_rows = []
    curve_rows = []
    branch_input_dims = get_branch_input_dims(branch_cols)

    for fold_idx, (tr_idx, test_idx) in enumerate(skf.split(np.zeros(len(y)), y), start=1):
        outer_train_df = dev_df.iloc[tr_idx].reset_index(drop=True)
        outer_test_df = dev_df.iloc[test_idx].reset_index(drop=True)

        inner_train_df, inner_val_df = make_train_earlystop_split(
            outer_train_df,
            val_frac=CV_EARLY_STOPPING_VAL_FRAC,
            seed=SEED + fold_idx,
        )

        X_train_dict, preprocessors = fit_preprocessors_and_make_tensors(inner_train_df, branch_cols)
        X_val_dict = transform_df_to_tensors(inner_val_df, branch_cols, preprocessors)
        X_test_dict = transform_df_to_tensors(outer_test_df, branch_cols, preprocessors)

        y_train = inner_train_df["label"].values.astype(np.float32)
        y_val = inner_val_df["label"].values.astype(int)
        y_test = outer_test_df["label"].values.astype(int)

        model, fold_curve_df, train_info = train_model(
            X_train_dict=X_train_dict,
            y_train=y_train,
            branch_input_dims=branch_input_dims,
            branch_config=branch_config,
            fusion_hidden_dims=fusion_hidden_dims,
            X_val_dict=X_val_dict,
            y_val=y_val,
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            min_epochs=MIN_EPOCHS,
        )

        test_probs = predict_probs(model, X_test_dict)
        metrics = bootstrap_binary_metrics(
            y_test,
            test_probs,
            threshold=THRESHOLD,
            n_bootstrap=BOOTSTRAP_N,
            ci=BOOTSTRAP_CI,
            seed=SEED + 1000 + fold_idx,
        )

        fold_rows.append({
            "fold": fold_idx,
            "n_outer_train": len(outer_train_df),
            "n_inner_train": len(inner_train_df),
            "n_inner_val": len(inner_val_df),
            "n_outer_test": len(outer_test_df),
            "best_epoch": train_info["best_epoch"],
            "best_val_loss": train_info["best_val_loss"],
            "best_val_auc_at_best_epoch": train_info["best_val_auc_at_best_epoch"],
            "stopped_epoch": train_info["stopped_epoch"],
            "n_bootstrap": metrics["n_bootstrap"],
            "bootstrap_ci": metrics["bootstrap_ci"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            "balanced_accuracy_point": metrics["balanced_accuracy_point"],
            "balanced_accuracy_bootstrap_std": metrics["balanced_accuracy_bootstrap_std"],
            "balanced_accuracy_bootstrap_ci_lower": metrics["balanced_accuracy_bootstrap_ci_lower"],
            "balanced_accuracy_bootstrap_ci_upper": metrics["balanced_accuracy_bootstrap_ci_upper"],
            "balanced_accuracy_bootstrap_n_valid": metrics["balanced_accuracy_bootstrap_n_valid"],
            "auc": metrics["auc"],
            "auc_point": metrics["auc_point"],
            "auc_bootstrap_std": metrics["auc_bootstrap_std"],
            "auc_bootstrap_ci_lower": metrics["auc_bootstrap_ci_lower"],
            "auc_bootstrap_ci_upper": metrics["auc_bootstrap_ci_upper"],
            "auc_bootstrap_n_valid": metrics["auc_bootstrap_n_valid"],
            "sensitivity": metrics["sensitivity"],
            "sensitivity_point": metrics["sensitivity_point"],
            "sensitivity_bootstrap_std": metrics["sensitivity_bootstrap_std"],
            "sensitivity_bootstrap_ci_lower": metrics["sensitivity_bootstrap_ci_lower"],
            "sensitivity_bootstrap_ci_upper": metrics["sensitivity_bootstrap_ci_upper"],
            "sensitivity_bootstrap_n_valid": metrics["sensitivity_bootstrap_n_valid"],
            "specificity": metrics["specificity"],
            "specificity_point": metrics["specificity_point"],
            "specificity_bootstrap_std": metrics["specificity_bootstrap_std"],
            "specificity_bootstrap_ci_lower": metrics["specificity_bootstrap_ci_lower"],
            "specificity_bootstrap_ci_upper": metrics["specificity_bootstrap_ci_upper"],
            "specificity_bootstrap_n_valid": metrics["specificity_bootstrap_n_valid"],
            "tp": metrics["tp"],
            "tn": metrics["tn"],
            "fp": metrics["fp"],
            "fn": metrics["fn"],
        })

        fold_curve_df = fold_curve_df.copy()
        fold_curve_df["fusion_hidden_dims"] = json.dumps(fusion_hidden_dims)
        fold_curve_df["fusion_head_label"] = head_label
        fold_curve_df["fold"] = fold_idx
        fold_curve_df["n_outer_train"] = len(outer_train_df)
        fold_curve_df["n_inner_train"] = len(inner_train_df)
        fold_curve_df["n_inner_val"] = len(inner_val_df)
        fold_curve_df["n_outer_test"] = len(outer_test_df)
        fold_curve_df["best_epoch"] = train_info["best_epoch"]
        fold_curve_df["best_val_loss"] = train_info["best_val_loss"]
        fold_curve_df["best_val_auc_at_best_epoch"] = train_info["best_val_auc_at_best_epoch"]
        fold_curve_df["stopped_epoch"] = train_info["stopped_epoch"]
        curve_rows.append(fold_curve_df)

        print(
            f"Fold {fold_idx}: "
            f"BA={metrics['balanced_accuracy']:.4f} "
            f"[{metrics['balanced_accuracy_bootstrap_ci_lower']:.4f}, {metrics['balanced_accuracy_bootstrap_ci_upper']:.4f}], "
            f"AUC={metrics['auc']:.4f} "
            f"[{metrics['auc_bootstrap_ci_lower']:.4f}, {metrics['auc_bootstrap_ci_upper']:.4f}], "
            f"Sen={metrics['sensitivity']:.4f} "
            f"[{metrics['sensitivity_bootstrap_ci_lower']:.4f}, {metrics['sensitivity_bootstrap_ci_upper']:.4f}], "
            f"Spe={metrics['specificity']:.4f} "
            f"[{metrics['specificity_bootstrap_ci_lower']:.4f}, {metrics['specificity_bootstrap_ci_upper']:.4f}], "
            f"best_epoch={train_info['best_epoch']}"
        )

    fold_df = pd.DataFrame(fold_rows)
    curves_df = pd.concat(curve_rows, ignore_index=True)

    ba_mean, ba_std = mean_std(fold_df["balanced_accuracy"])
    auc_mean, auc_std = mean_std(fold_df["auc"])
    sen_mean, sen_std = mean_std(fold_df["sensitivity"])
    spe_mean, spe_std = mean_std(fold_df["specificity"])

    summary = {
        "balanced_accuracy_mean": ba_mean,
        "balanced_accuracy_std": ba_std,
        "auc_mean": auc_mean,
        "auc_std": auc_std,
        "sensitivity_mean": sen_mean,
        "sensitivity_std": sen_std,
        "specificity_mean": spe_mean,
        "specificity_std": spe_std,
    }

    return summary, fold_df, curves_df


def retrain_best_on_full_dev_and_test_external(
    dev_df,
    external_df,
    branch_cols,
    branch_config,
    fusion_hidden_dims,
    head_label,
):
    branch_input_dims = get_branch_input_dims(branch_cols)

    retrain_train_df, retrain_val_df = make_train_earlystop_split(
        dev_df,
        val_frac=FINAL_RETRAIN_VAL_FRAC,
        seed=SEED,
    )

    print(f"Final retrain split -> train: {len(retrain_train_df)}, val: {len(retrain_val_df)}")
    print("Train label counts:")
    print(retrain_train_df["label"].value_counts().sort_index())
    print("Val label counts:")
    print(retrain_val_df["label"].value_counts().sort_index())

    X_train_dict, preprocessors = fit_preprocessors_and_make_tensors(retrain_train_df, branch_cols)
    X_val_dict = transform_df_to_tensors(retrain_val_df, branch_cols, preprocessors)
    X_ext_dict = transform_df_to_tensors(external_df, branch_cols, preprocessors)

    y_train = retrain_train_df["label"].values.astype(np.float32)
    y_val = retrain_val_df["label"].values.astype(int)
    y_ext = external_df["label"].values.astype(int)

    model, full_curve_df, retrain_info = train_model(
        X_train_dict=X_train_dict,
        y_train=y_train,
        branch_input_dims=branch_input_dims,
        branch_config=branch_config,
        fusion_hidden_dims=fusion_hidden_dims,
        X_val_dict=X_val_dict,
        y_val=y_val,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        min_epochs=MIN_EPOCHS,
    )

    internal_val_probs = predict_probs(model, X_val_dict)
    internal_val_metrics = compute_binary_metrics(y_val, internal_val_probs, threshold=THRESHOLD)

    ext_probs = predict_probs(model, X_ext_dict)
    ext_metrics = bootstrap_binary_metrics(
        y_ext,
        ext_probs,
        threshold=THRESHOLD,
        n_bootstrap=BOOTSTRAP_N,
        ci=BOOTSTRAP_CI,
        seed=SEED + 2000,
    )

    internal_val_results_df = pd.DataFrame([{
        "split": "DEV_internal_val",
        "n_train": len(retrain_train_df),
        "n_val": len(retrain_val_df),
        "best_epoch": retrain_info["best_epoch"],
        "best_val_loss_during_training": retrain_info["best_val_loss"],
        "best_val_auc_at_best_epoch": retrain_info["best_val_auc_at_best_epoch"],
        "stopped_epoch": retrain_info["stopped_epoch"],
        "balanced_accuracy": internal_val_metrics["balanced_accuracy"],
        "auc": internal_val_metrics["auc"],
        "sensitivity": internal_val_metrics["sensitivity"],
        "specificity": internal_val_metrics["specificity"],
        "tp": internal_val_metrics["tp"],
        "tn": internal_val_metrics["tn"],
        "fp": internal_val_metrics["fp"],
        "fn": internal_val_metrics["fn"],
        "fusion_hidden_dims": json.dumps(fusion_hidden_dims),
        "fusion_head_label": head_label,
    }])

    external_results_df = pd.DataFrame([{
        "site": EXTERNAL_SITE,
        "n_samples": len(external_df),
        "n_train_retrain": len(retrain_train_df),
        "n_val_retrain": len(retrain_val_df),
        "best_epoch": retrain_info["best_epoch"],
        "best_val_loss_during_training": retrain_info["best_val_loss"],
        "best_val_auc_at_best_epoch": retrain_info["best_val_auc_at_best_epoch"],
        "stopped_epoch": retrain_info["stopped_epoch"],
        "n_bootstrap": ext_metrics["n_bootstrap"],
        "bootstrap_ci": ext_metrics["bootstrap_ci"],
        "balanced_accuracy": ext_metrics["balanced_accuracy"],
        "balanced_accuracy_point": ext_metrics["balanced_accuracy_point"],
        "balanced_accuracy_bootstrap_std": ext_metrics["balanced_accuracy_bootstrap_std"],
        "balanced_accuracy_bootstrap_ci_lower": ext_metrics["balanced_accuracy_bootstrap_ci_lower"],
        "balanced_accuracy_bootstrap_ci_upper": ext_metrics["balanced_accuracy_bootstrap_ci_upper"],
        "balanced_accuracy_bootstrap_n_valid": ext_metrics["balanced_accuracy_bootstrap_n_valid"],
        "auc": ext_metrics["auc"],
        "auc_point": ext_metrics["auc_point"],
        "auc_bootstrap_std": ext_metrics["auc_bootstrap_std"],
        "auc_bootstrap_ci_lower": ext_metrics["auc_bootstrap_ci_lower"],
        "auc_bootstrap_ci_upper": ext_metrics["auc_bootstrap_ci_upper"],
        "auc_bootstrap_n_valid": ext_metrics["auc_bootstrap_n_valid"],
        "sensitivity": ext_metrics["sensitivity"],
        "sensitivity_point": ext_metrics["sensitivity_point"],
        "sensitivity_bootstrap_std": ext_metrics["sensitivity_bootstrap_std"],
        "sensitivity_bootstrap_ci_lower": ext_metrics["sensitivity_bootstrap_ci_lower"],
        "sensitivity_bootstrap_ci_upper": ext_metrics["sensitivity_bootstrap_ci_upper"],
        "sensitivity_bootstrap_n_valid": ext_metrics["sensitivity_bootstrap_n_valid"],
        "specificity": ext_metrics["specificity"],
        "specificity_point": ext_metrics["specificity_point"],
        "specificity_bootstrap_std": ext_metrics["specificity_bootstrap_std"],
        "specificity_bootstrap_ci_lower": ext_metrics["specificity_bootstrap_ci_lower"],
        "specificity_bootstrap_ci_upper": ext_metrics["specificity_bootstrap_ci_upper"],
        "specificity_bootstrap_n_valid": ext_metrics["specificity_bootstrap_n_valid"],
        "tp": ext_metrics["tp"],
        "tn": ext_metrics["tn"],
        "fp": ext_metrics["fp"],
        "fn": ext_metrics["fn"],
        "fusion_hidden_dims": json.dumps(fusion_hidden_dims),
        "fusion_head_label": head_label,
    }])

    external_pred_df = external_df[["id", "site", "label"]].copy()
    external_pred_df["prob"] = ext_probs
    external_pred_df["pred"] = (ext_probs >= THRESHOLD).astype(int)
    external_pred_df["fusion_hidden_dims"] = json.dumps(fusion_hidden_dims)
    external_pred_df["fusion_head_label"] = head_label

    full_curve_df = full_curve_df.copy()
    full_curve_df["fusion_hidden_dims"] = json.dumps(fusion_hidden_dims)
    full_curve_df["fusion_head_label"] = head_label
    full_curve_df["train_cohort"] = "DEV_train_80pct"
    full_curve_df["val_cohort"] = "DEV_val_20pct"
    full_curve_df["test_cohort"] = EXTERNAL_SITE
    full_curve_df["n_train"] = len(retrain_train_df)
    full_curve_df["n_val"] = len(retrain_val_df)
    full_curve_df["n_test"] = len(external_df)
    full_curve_df["best_epoch"] = retrain_info["best_epoch"]
    full_curve_df["best_val_loss"] = retrain_info["best_val_loss"]
    full_curve_df["best_val_auc_at_best_epoch"] = retrain_info["best_val_auc_at_best_epoch"]
    full_curve_df["stopped_epoch"] = retrain_info["stopped_epoch"]

    split_info = {
        "dev_sites": DEV_SITES,
        "external_site": EXTERNAL_SITE,
        "n_dev_total": len(dev_df),
        "n_train": len(retrain_train_df),
        "n_val": len(retrain_val_df),
        "n_external": len(external_df),
        "val_fraction": FINAL_RETRAIN_VAL_FRAC,
    }

    print(
        f"{EXTERNAL_SITE}: "
        f"BA={ext_metrics['balanced_accuracy']:.4f} "
        f"[{ext_metrics['balanced_accuracy_bootstrap_ci_lower']:.4f}, {ext_metrics['balanced_accuracy_bootstrap_ci_upper']:.4f}], "
        f"AUC={ext_metrics['auc']:.4f} "
        f"[{ext_metrics['auc_bootstrap_ci_lower']:.4f}, {ext_metrics['auc_bootstrap_ci_upper']:.4f}], "
        f"Sen={ext_metrics['sensitivity']:.4f} "
        f"[{ext_metrics['sensitivity_bootstrap_ci_lower']:.4f}, {ext_metrics['sensitivity_bootstrap_ci_upper']:.4f}], "
        f"Spe={ext_metrics['specificity']:.4f} "
        f"[{ext_metrics['specificity_bootstrap_ci_lower']:.4f}, {ext_metrics['specificity_bootstrap_ci_upper']:.4f}]"
    )

    return (
        model,
        full_curve_df,
        internal_val_results_df,
        external_results_df,
        external_pred_df,
        retrain_info,
        split_info,
    )


# ================= REPORTING HELPERS =================
def print_cohort_info(df, name):
    print(f"{name}: n={len(df)}")
    if len(df) > 0:
        print(df["label"].value_counts(dropna=False).sort_index())
    print()


def print_feature_presence(df, cohort_name, branch_cols):
    print(f"Feature presence in {cohort_name}:")
    for modality in MODALITIES:
        rad_cols = branch_cols[modality]["rad"]
        deep_cols = branch_cols[modality]["deep"]

        rad_present = df[rad_cols].notna().any(axis=1).sum()
        deep_present = df[deep_cols].notna().any(axis=1).sum()

        print(
            f"  {modality}: rad {rad_present}/{len(df)}, deep {deep_present}/{len(df)}"
        )
    print()


def main():
    print(f"Using device: {DEVICE}")
    print(f"DEV sites: {DEV_SITES}")
    print(f"External site: {EXTERNAL_SITE}")

    allowed_sites = set(DEV_SITES + [EXTERNAL_SITE])

    labels_df = load_labels()
    labels_df = labels_df[labels_df["site"].isin(allowed_sites)].copy().reset_index(drop=True)

    dev_labels_df = labels_df[labels_df["site"].isin(DEV_SITES)].copy().reset_index(drop=True)
    ext_labels_df = labels_df[labels_df["site"] == EXTERNAL_SITE].copy().reset_index(drop=True)

    print_cohort_info(dev_labels_df, "DEV labels")
    print_cohort_info(ext_labels_df, f"{EXTERNAL_SITE} labels")

    merged_all_df, branch_cols = build_multimodal_table(labels_df)

    dev_merged_df = merged_all_df[merged_all_df["site"].isin(DEV_SITES)].copy().reset_index(drop=True)
    ext_merged_df = merged_all_df[merged_all_df["site"] == EXTERNAL_SITE].copy().reset_index(drop=True)

    print(f"Final merged DEV size:   {len(dev_merged_df)}")
    print(f"Final merged {EXTERNAL_SITE} size: {len(ext_merged_df)}")
    print()

    print_feature_presence(dev_merged_df, "DEV", branch_cols)
    print_feature_presence(ext_merged_df, EXTERNAL_SITE, branch_cols)

    print("Branch dims:")
    for modality in MODALITIES:
        print(
            f"{modality}: "
            f"rad_dim={len(branch_cols[modality]['rad'])}, "
            f"deep_dim={len(branch_cols[modality]['deep'])}, "
            f"rad_hidden={BRANCH_CONFIG[modality]['rad_hidden']}, "
            f"deep_hidden={BRANCH_CONFIG[modality]['deep_hidden']}"
        )

    print()
    print("Fusion head options to test:")
    for dims in FUSION_HEAD_OPTIONS:
        print(f"  {dims if dims else 'direct_to_1'}")
    print("=" * 80)

    all_results = []
    all_fold_results = []
    all_curve_results = []

    # ================= DEV CV MODEL SELECTION =================
    for fusion_hidden_dims in FUSION_HEAD_OPTIONS:
        head_label = fusion_head_to_label(fusion_hidden_dims)
        safe_head_label = sanitize_for_filename(head_label)

        print(f"Testing fusion head: {head_label}")

        summary, fold_df, curves_df = train_eval_multimodal_cv(
            dev_df=dev_merged_df,
            branch_cols=branch_cols,
            branch_config=BRANCH_CONFIG,
            fusion_hidden_dims=fusion_hidden_dims,
            head_label=head_label,
        )

        result_row = {
            "dev_sites": json.dumps(DEV_SITES),
            "external_site": EXTERNAL_SITE,
            "n_samples_dev": len(dev_merged_df),
            "n_samples_external": len(ext_merged_df),

            "adc_rad_dim": len(branch_cols["ADC"]["rad"]),
            "adc_deep_dim": len(branch_cols["ADC"]["deep"]),
            "t1_rad_dim": len(branch_cols["T1"]["rad"]),
            "t1_deep_dim": len(branch_cols["T1"]["deep"]),
            "t2_rad_dim": len(branch_cols["T2"]["rad"]),
            "t2_deep_dim": len(branch_cols["T2"]["deep"]),

            "adc_rad_hidden": BRANCH_CONFIG["ADC"]["rad_hidden"],
            "adc_deep_hidden": BRANCH_CONFIG["ADC"]["deep_hidden"],
            "t1_rad_hidden": BRANCH_CONFIG["T1"]["rad_hidden"],
            "t1_deep_hidden": BRANCH_CONFIG["T1"]["deep_hidden"],
            "t2_rad_hidden": BRANCH_CONFIG["T2"]["rad_hidden"],
            "t2_deep_hidden": BRANCH_CONFIG["T2"]["deep_hidden"],

            "fusion_hidden_dims": json.dumps(fusion_hidden_dims),
            "fusion_head_label": head_label,

            "balanced_accuracy_mean": summary["balanced_accuracy_mean"],
            "balanced_accuracy_std": summary["balanced_accuracy_std"],
            "auc_mean": summary["auc_mean"],
            "auc_std": summary["auc_std"],
            "sensitivity_mean": summary["sensitivity_mean"],
            "sensitivity_std": summary["sensitivity_std"],
            "specificity_mean": summary["specificity_mean"],
            "specificity_std": summary["specificity_std"],
        }
        all_results.append(result_row)

        fold_df = fold_df.copy()
        fold_df["fusion_hidden_dims"] = json.dumps(fusion_hidden_dims)
        fold_df["fusion_head_label"] = head_label
        all_fold_results.append(fold_df)

        curves_df = curves_df.copy()
        all_curve_results.append(curves_df)

        curve_csv = os.path.join(LEARNING_CURVE_CSV_DIR, f"learning_curve_{safe_head_label}.csv")
        curves_df.to_csv(curve_csv, index=False)

        print(f"  BA:  {summary['balanced_accuracy_mean']:.4f} ± {summary['balanced_accuracy_std']:.4f}")
        print(f"  AUC: {summary['auc_mean']:.4f} ± {summary['auc_std']:.4f}")
        print(f"  Sen: {summary['sensitivity_mean']:.4f} ± {summary['sensitivity_std']:.4f}")
        print(f"  Spe: {summary['specificity_mean']:.4f} ± {summary['specificity_std']:.4f}")
        print()

    results_df = pd.DataFrame(all_results)
    fold_results_df = pd.concat(all_fold_results, ignore_index=True)
    learning_curve_df = pd.concat(all_curve_results, ignore_index=True)

    summary_csv = os.path.join(OUTPUT_DIR, "multimodal_fusion_summary.csv")
    fold_csv = os.path.join(OUTPUT_DIR, "multimodal_fusion_fold_results.csv")
    learning_curve_csv = os.path.join(OUTPUT_DIR, "multimodal_fusion_learning_curves_all.csv")

    results_df.to_csv(summary_csv, index=False)
    fold_results_df.to_csv(fold_csv, index=False)
    learning_curve_df.to_csv(learning_curve_csv, index=False)

    ba_plot = os.path.join(PLOT_DIR, "fusion_head_balanced_accuracy_barplot.png")
    auc_plot = os.path.join(PLOT_DIR, "fusion_head_auc_barplot.png")

    plot_metric_bar(
        results_df=results_df,
        mean_col="balanced_accuracy_mean",
        std_col="balanced_accuracy_std",
        ylabel="Balanced Accuracy",
        title="(a) Multi-Modal Imaging Features",
        out_path=ba_plot,
    )

    plot_metric_bar(
        results_df=results_df,
        mean_col="auc_mean",
        std_col="auc_std",
        ylabel="AUROC",
        title="(b) Multi-Modal Imaging Features",
        out_path=auc_plot,
    )

    # ================= PICK BEST MODEL =================
    results_df_sorted = results_df.sort_values(
        ["balanced_accuracy_mean", "auc_mean"],
        ascending=False,
    ).reset_index(drop=True)

    best_row = results_df_sorted.iloc[0].to_dict()
    best_fusion_hidden_dims = json.loads(best_row["fusion_hidden_dims"])
    best_head_label = best_row["fusion_head_label"]

    best_json = os.path.join(OUTPUT_DIR, "multimodal_fusion_best.json")
    with open(best_json, "w") as f:
        json.dump(best_row, f, indent=2)

    print("=" * 80)
    print("Best fusion head selected from DEV CV:")
    print(json.dumps(best_row, indent=2))
    print("=" * 80)

    # ================= RETRAIN ON DEV, TEST ONLY ON NYU =================
    (
        _,
        final_curve_df,
        internal_val_results_df,
        external_results_df,
        external_pred_df,
        retrain_info,
        split_info,
    ) = retrain_best_on_full_dev_and_test_external(
        dev_df=dev_merged_df,
        external_df=ext_merged_df,
        branch_cols=branch_cols,
        branch_config=BRANCH_CONFIG,
        fusion_hidden_dims=best_fusion_hidden_dims,
        head_label=best_head_label,
    )

    external_results_csv = os.path.join(OUTPUT_DIR, "best_model_external_results.csv")
    external_pred_csv = os.path.join(OUTPUT_DIR, "best_model_external_predictions.csv")
    internal_val_csv = os.path.join(OUTPUT_DIR, "best_model_internal_val_results.csv")
    final_curve_csv = os.path.join(OUTPUT_DIR, "best_model_dev_retrain_learning_curve.csv")
    final_curve_png = os.path.join(FINAL_RETRAIN_DIR, "best_model_dev_retrain_learning_curve.png")
    final_curve_pdf = os.path.join(FINAL_RETRAIN_DIR, "best_model_dev_retrain_learning_curve.pdf")

    external_results_df.to_csv(external_results_csv, index=False)
    external_pred_df.to_csv(external_pred_csv, index=False)
    internal_val_results_df.to_csv(internal_val_csv, index=False)
    final_curve_df.to_csv(final_curve_csv, index=False)

    plot_learning_curve(
        curve_df=final_curve_df,
        out_path_png=final_curve_png,
        out_path_pdf=final_curve_pdf,
        title=None,
        panel_label=None,
    )

    best_with_external = {
        "best_dev_cv_model": best_row,
        "final_retrain_split": split_info,
        "final_retrain_info": retrain_info,
        "internal_val_results": internal_val_results_df.to_dict(orient="records"),
        "external_results": external_results_df.to_dict(orient="records"),
        "final_curve_csv": final_curve_csv,
        "internal_val_results_csv": internal_val_csv,
        "external_results_csv": external_results_csv,
        "external_predictions_csv": external_pred_csv,
    }

    best_with_external_json = os.path.join(OUTPUT_DIR, "multimodal_fusion_best_with_external.json")
    with open(best_with_external_json, "w") as f:
        json.dump(best_with_external, f, indent=2)

    print("=" * 80)
    print("DONE")
    print(f"Saved summary:                    {summary_csv}")
    print(f"Saved folds:                      {fold_csv}")
    print(f"Saved best json:                  {best_json}")
    print(f"Saved best+external json:         {best_with_external_json}")
    print(f"Saved BA plot:                    {ba_plot}")
    print(f"Saved AUC plot:                   {auc_plot}")
    print(f"Saved CV learning curve CSV:      {learning_curve_csv}")
    print(f"Saved curve CSV folder:           {LEARNING_CURVE_CSV_DIR}")
    print(f"Saved internal val results CSV:   {internal_val_csv}")
    print(f"Saved external results CSV:       {external_results_csv}")
    print(f"Saved external predictions CSV:   {external_pred_csv}")
    print(f"Saved final retrain curve CSV:    {final_curve_csv}")
    print(f"Saved final retrain curve PNG:    {final_curve_png}")
    print(f"Saved final retrain curve PDF:    {final_curve_pdf}")
    print()

    print("All DEV CV results:")
    print(results_df_sorted.to_string(index=False))
    print()

    print("Internal validation results from final DEV retraining split:")
    print(internal_val_results_df.to_string(index=False))
    print()

    print(f"External test results on {EXTERNAL_SITE} using best DEV CV model retrained with 80/20 DEV split and early stopping on val loss:")
    print(external_results_df.to_string(index=False))


if __name__ == "__main__":
    main()

Using device: cuda
DEV sites: ['CCHMC', 'WISC', 'NYU']
External site: MICH
DEV labels: n=472
label
0    221
1    251
Name: count, dtype: int64

MICH labels: n=46
label
0    25
1    21
Name: count, dtype: int64

Final merged DEV size:   472
Final merged MICH size: 46

Feature presence in DEV:
  ADC: rad 149/472, deep 225/472
  T1: rad 309/472, deep 331/472
  T2: rad 424/472, deep 440/472

Feature presence in MICH:
  ADC: rad 39/46, deep 43/46
  T1: rad 21/46, deep 46/46
  T2: rad 31/46, deep 31/46

Branch dims:
ADC: rad_dim=200, deep_dim=1152, rad_hidden=16, deep_hidden=64
T1: rad_dim=200, deep_dim=1152, rad_hidden=8, deep_hidden=128
T2: rad_dim=200, deep_dim=1152, rad_hidden=8, deep_hidden=128

Fusion head options to test:
  [32]
  [16]
  [8]
  direct_to_1
Testing fusion head: 32
Early stopping at epoch 62 | best epoch = 47 | best val loss = 0.516673
Fold 1: BA=0.7050 [0.6064, 0.7988], AUC=0.7321 [0.6317, 0.8263], Sen=0.7822 [0.6667, 0.8868], Spe=0.6277 [0.4886, 0.7848], best_epoch=47
